# 16 — Unified Recommendations (consumer demo)

Qualitative demonstration of the nine-signal unified POI recommender. Downstream of the profile card (NB 15). Headline evaluation is in NB 17; signal-level diagnostics in NB 18.

**Signals** (weighted — see `UNIFIED_WEIGHTS`):
1. `archetype_sim` — OWL profile class similarity (IC-weighted Jaccard)
2. `semantic_sim` — behavioral feature cosine
3. `route_sim` — Jaccard POI overlap + category sequence LCS
4. `flow_prob` — recency-weighted transition probability
5. `dominant_ctx` — co-visited with dominant POI (SR-05)
6. `user_pref` — category preference alignment (SR-20)
7. `significant` — notable missed POIs
8. `engagement` — engagement-level match (SR-10)

**4-layer explainability**: signal decomposition → archetype reasoning → rule endorsement (SR-30..34) → KG provenance.

**Prerequisites:**
- Fuseki running with enriched dataset
- `python scripts/run_owl_classification.py` (inferred graph materialized)
- SR-05, SR-06, SR-10, SR-20 materialized
- Semantic features in `notebooks/output/semantic_features/trajectory_features.json`

**Not an evaluation notebook.** All numbers below describe the stack running on hand-picked trajectories. Do not quote them.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path
from IPython.display import display as ipydisplay

from helpers.unified_recommender import UnifiedRecommender, UNIFIED_WEIGHTS, _RULE_LABELS
from helpers.ontological_recommender import LOW_IC_THRESHOLD
from helpers.map_visualizer import TrajectoMapVisualizer

FEATURES_PATH = Path('./output/semantic_features/trajectory_features.json')

rec = UnifiedRecommender()
rec.load_all_data(features_path=str(FEATURES_PATH))

visualizer = TrajectoMapVisualizer(
    default_location=(41.9028, 12.4964),
    default_zoom=13
)

print(f"\nSignal weights: {UNIFIED_WEIGHTS}")

## 1. Select Test Trajectories

A diverse set: one trajectory per dominant archetype class.

In [ ]:
from collections import defaultdict

def most_specific_class(uri):
    classes = rec.classifications.get(uri, set())
    informative = [(c, rec.ic_weights.get(c, 0)) for c in classes
                   if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD]
    return max(informative, key=lambda x: x[1]) if informative else ('', 0)

by_archetype = defaultdict(list)
for uri in rec.classifications:
    cls, ic = most_specific_class(uri)
    if cls:
        by_archetype[cls].append((uri, ic))

test_trajs = []
seen = set()
for cls in sorted(by_archetype, key=lambda c: -rec.ic_weights.get(c, 0)):
    if len(test_trajs) >= 10:
        break
    if cls in seen:
        continue
    candidates = by_archetype[cls]
    for uri, ic in sorted(candidates, key=lambda x: -len(rec.classifications[x[0]])):
        pois = rec.traj_poi_sets.get(uri, set())
        if len(pois) >= 3:
            test_trajs.append(uri)
            seen.add(cls)
            break

print(f"Test trajectories: {len(test_trajs)}\n")
for uri in test_trajs:
    name = uri.split('#')[-1]
    cls, ic = most_specific_class(uri)
    n_classes = sum(1 for c in rec.classifications[uri]
                    if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD)
    n_pois = len(rec.traj_poi_sets.get(uri, set()))
    print(f"  {name}: {cls} (IC={ic:.2f}), {n_classes} informative classes, {n_pois} POIs")

## 2. Unified Recommendations — Full 4-Layer Display

Each recommendation with signal decomposition, archetype reasoning, rule endorsement, and KG provenance.

In [ ]:
all_recs = {}

def _prov_detail(p):
    """Format signal-specific provenance values."""
    sig = p['signal']
    if sig == 'archetype_sim':
        parts = []
        if 'top_similarity' in p:
            parts.append(f"top_sim={p['top_similarity']:.3f}")
        if 'n_sources' in p:
            parts.append(f"{p['n_sources']} sources")
        return ', '.join(parts) if parts else ''
    if sig == 'semantic_sim':
        parts = []
        if 'top_cosine' in p:
            parts.append(f"top_cosine={p['top_cosine']:.3f}")
        if 'n_contributors' in p:
            parts.append(f"{p['n_contributors']} contributors")
        return ', '.join(parts) if parts else ''
    if sig == 'rule_coverage':
        rules = p.get('rules', [])
        return ', '.join(rules) if rules else ''
    if sig == 'dominant_ctx':
        parts = []
        if 'dominant_poi' in p:
            parts.append(p['dominant_poi'])
        if 'covisit_weight' in p:
            parts.append(f"weight={p['covisit_weight']:.1f}")
        return ', '.join(parts) if parts else ''
    if sig == 'user_pref':
        parts = []
        if 'category' in p:
            parts.append(p['category'])
        if 'pref_score' in p:
            parts.append(f"pref={p['pref_score']:.2f}")
        return ', '.join(parts) if parts else ''
    if sig == 'engagement':
        if 'match_count' in p:
            return f"{p['match_count']} matches"
    if sig == 'significant':
        if 'total_significant' in p:
            return f"{p['total_significant']} significant POIs"
    return ''

for traj_uri in test_trajs:
    traj_name = traj_uri.split('#')[-1]
    cls, ic = most_specific_class(traj_uri)
    informative = sorted(
        [c for c in rec.classifications[traj_uri]
         if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD],
        key=lambda c: -rec.ic_weights.get(c, 0)
    )

    recs = rec.recommend(traj_uri, top_k=5)
    all_recs[traj_uri] = recs

    print(f"\n{'='*80}")
    print(f"{traj_name}")
    print(f"  Profile: {', '.join(informative[:5])}")
    if len(informative) > 5:
        print(f"           + {len(informative) - 5} more")

    for rank, r in enumerate(recs[:5], 1):
        signals = r['signals']
        active = sum(1 for v in signals.values() if v > 0)

        print(f"\n  {'—'*70}")
        print(f"  #{rank}  {r['label'][:35]:<37} {r['category']:<18} "
              f"score={r['score']:.3f}  [{r['confidence']}]")

        # Layer 1: Signal decomposition
        print(f"\n  LAYER 1 — Signal Decomposition ({active}/7 signals active)")
        weights = UNIFIED_WEIGHTS
        for sig in sorted(signals, key=lambda s: -weights.get(s, 0) * signals[s]):
            val = signals[sig]
            w = weights.get(sig, 0)
            contrib = w * val
            if contrib > 0:
                bar = '\u2588' * max(1, int(contrib * 80))
                print(f"    {sig:<16} {contrib:.3f}  {bar}")
            elif val > 0:
                print(f"    {sig:<16} {contrib:.3f}")

        # Layer 2: Archetype reasoning
        archetype = r.get('archetype')
        print("\n  LAYER 2 — Archetype Reasoning")
        if archetype:
            shared = ', '.join(archetype['shared_classes'])
            print(f"    Profile link:   {shared}")
            print(f"                    shared by {archetype['source_count']} similar trajectories")
            if archetype.get('profile_fit'):
                fit = ', '.join(archetype['profile_fit'])
                print(f"    Profile fit:    {fit}")
                print(f"                    aligns with {r['category']} (alignment={archetype['alignment']:.2f})")
            else:
                print("    Profile fit:    No direct class-to-category match")
        else:
            print("    No archetype signal for this POI")

        # Layer 3: Rule endorsement
        rules = r.get('rules')
        print("\n  LAYER 3 — Rule Endorsement")
        if rules:
            print(f"    {rules['count']}/5 rules endorse:")
            for rule_name, explanation in rules['explanations'].items():
                label = _RULE_LABELS.get(rule_name, rule_name)
                print(f"      {rule_name} ({label}): {explanation}")
        else:
            print("    No SR rules matched this POI")

        # Layer 4: KG provenance
        provenance = r.get('provenance', [])
        print("\n  LAYER 4 — KG Provenance")
        if provenance:
            for p in provenance:
                detail = _prov_detail(p)
                pred = p.get('predicate', '')
                pred_short = pred[:52]
                if detail:
                    print(f"    {p['signal']:<16} \u2192  {pred_short:<54} {detail}")
                else:
                    print(f"    {p['signal']:<16} \u2192  {pred_short}")
        else:
            print("    No provenance data")

## 3. Signal Contribution Analysis

Aggregate signal contributions across all recommendations.

In [ ]:
all_flat = [r for recs in all_recs.values() for r in recs]
if not all_flat:
    print("No recommendations generated.")
else:
    signal_names = list(UNIFIED_WEIGHTS.keys())
    n_recs = len(all_flat)

    # Mean weighted contribution per signal
    mean_contribs = {}
    for sig in signal_names:
        w = UNIFIED_WEIGHTS[sig]
        vals = [r['signals'].get(sig, 0) * w for r in all_flat]
        mean_contribs[sig] = np.mean(vals)

    # Active signal frequency
    active_freq = {sig: sum(1 for r in all_flat if r['signals'].get(sig, 0) > 0)
                   for sig in signal_names}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Mean contribution
    sorted_sigs = sorted(signal_names, key=lambda s: -mean_contribs[s])
    axes[0].barh(sorted_sigs, [mean_contribs[s] for s in sorted_sigs], color='#4C72B0')
    axes[0].set_xlabel('Mean Weighted Contribution')
    axes[0].set_title('Signal Contribution (mean across all recs)')
    axes[0].invert_yaxis()

    # Active frequency
    axes[1].barh(sorted_sigs, [active_freq[s] / n_recs * 100 for s in sorted_sigs],
                 color='#DD8452')
    axes[1].set_xlabel('% of Recommendations')
    axes[1].set_title('Signal Activation Frequency')
    axes[1].invert_yaxis()

    # Confidence distribution
    conf_counts = Counter(r['confidence'] for r in all_flat)
    conf_labels = ['strong', 'moderate', 'weak']
    conf_vals = [conf_counts.get(c, 0) for c in conf_labels]
    colors = ['#2ca02c', '#ff7f0e', '#d62728']
    axes[2].bar(conf_labels, conf_vals, color=colors)
    axes[2].set_ylabel('Count')
    axes[2].set_title('Recommendation Confidence')

    plt.tight_layout()
    plt.show()

    # Summary stats
    scores = [r['score'] for r in all_flat]
    cats = Counter(r['category'] for r in all_flat)
    aligned = sum(1 for r in all_flat
                  if r.get('archetype') and r['archetype']['alignment'] > 0)
    ruled = sum(1 for r in all_flat if r.get('rules'))

    print(f"Total recommendations: {n_recs} across {len(test_trajs)} trajectories")
    print(f"Score range: [{min(scores):.3f}, {max(scores):.3f}]")
    print(f"Mean score: {np.mean(scores):.3f}")
    print(f"Profile-aligned: {aligned}/{n_recs} ({100*aligned/n_recs:.0f}%)")
    print(f"Rule-endorsed: {ruled}/{n_recs} ({100*ruled/n_recs:.0f}%)")
    print(f"Mean active signals: {np.mean([sum(1 for v in r['signals'].values() if v > 0) for r in all_flat]):.1f}/7")
    print(f"Distinct categories: {len(cats)}")
    print(f"Distinct POIs: {len(set(r['poi'] for r in all_flat))}")

## 4. Per-Trajectory Breakdown

In [ ]:
for traj_uri in test_trajs:
    recs = all_recs[traj_uri]
    name = traj_uri.split('#')[-1][:35]
    if not recs:
        print(f"{name}: 0 recs")
        continue

    scores = [r['score'] for r in recs]
    cats = set(r['category'] for r in recs)
    n_aligned = sum(1 for r in recs
                    if r.get('archetype') and r['archetype']['alignment'] > 0)
    n_ruled = sum(1 for r in recs if r.get('rules'))
    mean_active = np.mean([sum(1 for v in r['signals'].values() if v > 0)
                           for r in recs])

    print(f"{name}:")
    print(f"  {len(recs)} recs, {len(cats)} categories, "
          f"score=[{min(scores):.3f}, {max(scores):.3f}]")
    print(f"  {n_aligned} profile-aligned, {n_ruled} rule-endorsed, "
          f"{mean_active:.1f}/7 mean active signals")
    print()

## 5. Recommendation Maps

Visited POIs and unified recommendations on interactive maps.

In [ ]:
MAPS_DIR = Path('output/maps')
MAPS_DIR.mkdir(parents=True, exist_ok=True)

for traj_uri in test_trajs:
    traj_name = traj_uri.split('#')[-1]
    recs = all_recs[traj_uri]

    # Build dwell/interaction lookup for this trajectory
    interactions = rec.interaction_data.get(traj_uri, [])
    poi_dwell = {}  # poi_uri -> (dwell_seconds, int_type)
    for item in interactions:
        poi_dwell[item['poi']] = (item.get('dwell', 0), item.get('int_type', ''))

    # Visited POIs for this trajectory (with dwell + interaction)
    visited_pois = []
    for poi_uri in rec.sequences.get(traj_uri, []):
        pm = rec.poi_metadata.get(poi_uri, {})
        if pm.get('lat') and pm.get('lon'):
            dwell, int_type = poi_dwell.get(poi_uri, (0, ''))
            visited_pois.append({
                'poi': poi_uri,
                'label': pm.get('label', '?'),
                'latitude': float(pm['lat']),
                'longitude': float(pm['lon']),
                'category': pm.get('category', ''),
                'dwell_seconds': dwell,
                'interaction_type': int_type,
            })

    # Enriched recommendations
    enriched = []
    for r in recs:
        if r['latitude'] and r['longitude']:
            enriched.append({
                'poi': r['poi'],
                'label': r['label'],
                'latitude': r['latitude'],
                'longitude': r['longitude'],
                'category': r['category'],
                'score': r['score'],
                'explanation': r['explanation'],
            })

    # Build trajectory info for the expandable panel
    meta = rec.traj_metadata.get(traj_uri, {})
    owl_classes = rec.classifications.get(traj_uri, set())
    informative_classes = sorted(
        [(c, rec.ic_weights.get(c, 0)) for c in owl_classes
         if rec.ic_weights.get(c, 0) >= LOW_IC_THRESHOLD],
        key=lambda x: -x[1]
    )

    trajectory_info = {
        'user': meta.get('user', ''),
        'purposes': meta.get('purposes', set()),
        'nav_styles': meta.get('nav_styles', set()),
        'engagement': meta.get('engagement', ''),
        'n_visited': len(visited_pois),
        'n_recommendations': len(enriched),
        'classes': informative_classes,
    }

    print(f"\n{'='*50}")
    print(f"{traj_name}: {len(visited_pois)} visited, {len(enriched)} recommendations")

    if enriched and visited_pois:
        rec_map = visualizer.create_recommendation_map(
            visited_pois,
            enriched,
            title=f"Unified Recs: {traj_name[:25]}",
            show_explanations=True,
            trajectory_info=trajectory_info,
        )
        map_path = MAPS_DIR / f'nb16_unified_{traj_name}.html'
        rec_map.save(str(map_path))
        print(f"Blue = Visited | Green = Recommended  |  Saved: {map_path}")
        ipydisplay(rec_map)
    else:
        print("Insufficient location data for map")
